# Design a binder to a real protein

**Protein design practical — University of Tartu**

You will pick a target protein, choose where on it you want to bind, and run
**BoltzGen** — a 2025 model that generates a binder, designs its sequence,
folds the result, and scores it, all in one pipeline.

Then you will read the scorecard it produces and decide whether you would
actually make any of them.

---

## ⚠️ Do this first

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Run the next cell now.** It installs BoltzGen and downloads ~6 GB of
   model weights. Start it and leave it running while we talk.

A design run takes about ten minutes, so you will get two or three attempts.
Choose your target and binding site thoughtfully rather than guessing.


In [ ]:
#@title ⚙️ Install BoltzGen and fetch weights  (~10 min, run once)
%%time
import os, subprocess, sys, textwrap

# Colab is Python 3.13; boltzgen pins numpy==2.0.2 which has no 3.13 wheels.
# BoltzGen is a CLI, so give it a private 3.12 interpreter via uv.
VENV = '/content/bgenv'
BG   = f'{VENV}/bin/boltzgen'
os.environ['HF_HOME'] = '/content/hf'

def sh(cmd, **kw):
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if not os.path.exists(BG):
    sh(f'{sys.executable} -m pip install -q uv')
    sh(f'{sys.executable} -m uv venv --python 3.12 {VENV}')
    sh(f'{sys.executable} -m uv pip install --python {VENV}/bin/python -q boltzgen')

# does the private interpreter see the GPU?
sh(f'{VENV}/bin/python -c "import torch,sys;'
   f'print(\'torch\', torch.__version__, \'| cuda:\', torch.cuda.is_available());'
   f'print(\'gpu:\', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"')

# pre-fetch weights now rather than stalling in the middle of the session
print('\nfetching model weights (~6 GB, once) ...')
sh(f'{sys.executable} -m pip install -q huggingface_hub py3Dmol pandas')
from huggingface_hub import snapshot_download
snapshot_download('boltzgen/boltzgen-1', cache_dir='/content/hf/hub')
snapshot_download('boltzgen/inference-data', repo_type='dataset', cache_dir='/content/hf/hub')

sh(f'{BG} --help > /dev/null')
print('\n✅ BoltzGen installed and weights cached.')


---
## 1 · Choose your target

Small targets run fastest. The structure is downloaded from the PDB and
cleaned — one chain, protein atoms only, no waters or ligands — because
BoltzGen expects a tidy single chain.

The cell prints the sequence and the residue numbering you will need in a moment.


In [ ]:
#@title 🎯 Fetch and clean the target  { display-mode: "form" }
target = "Ubiquitin (1UBQ, 76 aa)" #@param ["Crambin (1CRN, 46 aa — fastest)", "Protein G B1 domain (1PGA, 56 aa)", "Ubiquitin (1UBQ, 76 aa)", "Lysozyme (4LZT, 129 aa — slowest)"]
keep_chain = "A" #@param {type:"string"}

import urllib.request, os
TARGETS = {
        "Crambin (1CRN, 46 aa \u2014 fastest)": "1CRN",
        "Protein G B1 domain (1PGA, 56 aa)": "1PGA",
        "Ubiquitin (1UBQ, 76 aa)": "1UBQ",
        "Lysozyme (4LZT, 129 aa \u2014 slowest)": "4LZT"
    }
pdb_id = TARGETS[target]

raw = f'/content/{pdb_id}.pdb'
if not os.path.exists(raw):
    urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', raw)

THREE2ONE = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
             'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
             'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

clean, seq, resids, seen = [], '', [], set()
n = 0
for l in open(raw):
    if not l.startswith('ATOM'):
        continue
    if l[21] != keep_chain:
        continue
    if l[16] not in (' ', 'A'):          # drop alternate locations
        continue
    if l[17:20].strip() not in THREE2ONE:
        continue
    n += 1
    clean.append(f'{l[:6]}{n:5d}{l[11:16]} {l[17:]}'.rstrip() + '\n')
    key = l[22:27]
    if key not in seen:
        seen.add(key)
        seq += THREE2ONE[l[17:20].strip()]
        resids.append(int(l[22:26]))

TARGET_PDB = '/content/target.pdb'
open(TARGET_PDB, 'w').writelines(clean + ['END\n'])

print(f'{target}  ->  chain {keep_chain}, {len(resids)} residues, {n} atoms')
print(f'residue numbers run {resids[0]} to {resids[-1]}\n')
print('sequence:')
for i in range(0, len(seq), 60):
    print(f'  {resids[i]:>4d}  {seq[i:i+60]}')


---
## 2 · Choose where to bind, and set the run going

Pick a **hotspot residue** — a residue number from the list above that you want
your binder to touch. Choose it for a reason: a known functional site, an
exposed patch, something you can justify.

> **A warning worth having in advance.** This hotspot is a *soft* preference, not
> a constraint. We tested it on this cluster: with a block that had one obviously
> bindable groove, binders went to that groove regardless of which residue we
> named. So check afterwards whether the binder actually went where you asked —
> the last cell does exactly that. Do not assume it obeyed.

`num_designs` is how many candidates to generate, `budget` how many survive
filtering. Keep them small — this is roughly ten minutes as configured.


In [ ]:
#@title 🧬 Run BoltzGen  (~10 minutes)
%%time
hotspot_residue = 44 #@param {type:"integer"}
binder_length_min = 50 #@param {type:"slider", min:20, max:90, step:5}
binder_length_max = 70 #@param {type:"slider", min:30, max:110, step:5}
num_designs = 4 #@param ["2", "4", "8"] {type:"raw"}
budget = 2 #@param ["1", "2", "4"] {type:"raw"}
run_name = "run1" #@param {type:"string"}

import subprocess, os, textwrap
assert hotspot_residue in resids, (
    f'Residue {hotspot_residue} is not in this target '
    f'(it runs {resids[0]}-{resids[-1]}). Pick one from the list above.')
assert binder_length_max > binder_length_min, 'max must exceed min'

OUT = f'/content/out_{run_name}'
yaml = textwrap.dedent(f'''\
    entities:
      - file:
          path: {TARGET_PDB}
          include:
            - chain:
                id: {keep_chain}
          binding_types:
           - chain:
               id: {keep_chain}
               binding: {hotspot_residue}

      - protein:
          id: C
          sequence: {binder_length_min}..{binder_length_max}
    ''')
open('/content/design.yaml', 'w').write(yaml)
print(yaml)

env = dict(os.environ, HF_HOME='/content/hf')
cmd = (f'{BG} run /content/design.yaml --output {OUT} '
       f'--protocol protein-anything --inverse_fold_avoid "" '
       f'--num_designs {num_designs} --budget {budget} --devices 1')
print('$', cmd, '\n')
subprocess.run(cmd, shell=True, env=env, check=True)
print(f'\n✅ done -> {OUT}')


---
## 3 · The scorecard

BoltzGen writes one row per design with around 200 columns. These are the ones
that decide whether a design is worth making:

| Column | Meaning | Want |
|---|---|---|
| `design_to_target_iptm` | confidence in the interface | higher; >0.8 is strong |
| `min_design_to_target_pae` | expected error across the interface, Å | lower; <5 is promising |
| `bb_rmsd` | did the designed sequence refold to the designed shape | <2 Å |
| `liability_*` | chemical problems that make a molecule unmakeable | zero |

**Read the liabilities before you get excited about the ipTM.** A design can
score well on binding and still be impossible to produce.


In [ ]:
#@title 📊 Read the scorecard
import glob, os, subprocess, sys
try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=True)
    import pandas as pd
pd.set_option('display.width', 200)

csv = f'{OUT}/final_ranked_designs/all_designs_metrics.csv'
if not os.path.exists(csv):
    print('No results - run the previous cell first.')
else:
    df = pd.read_csv(csv)
    cols = [c for c in ['id', 'final_rank', 'design_to_target_iptm',
                        'min_design_to_target_pae', 'bb_rmsd',
                        'liability_num_violations', 'liability_violations_summary',
                        'designed_sequence'] if c in df.columns]
    df = df.sort_values('design_to_target_iptm', ascending=False)
    display(df[cols])

    best = df.iloc[0]
    print(f"\nBest by interface confidence: {best['id']}")
    print(f"  ipTM          {best['design_to_target_iptm']:.3f}", end='   ')
    print('strong' if best['design_to_target_iptm'] > 0.8 else
          'ambiguous' if best['design_to_target_iptm'] > 0.6 else 'weak')
    print(f"  interface pAE {best['min_design_to_target_pae']:.2f} A")
    v = best.get('liability_violations_summary')
    if isinstance(v, str) and v.strip():
        print(f"  ⚠️ liabilities: {v}")
        print('     -> this molecule has chemical problems regardless of its score')
    else:
        print('  no developability liabilities flagged')
    print(f"\n  sequence: {best['designed_sequence']}")


---
## 4 · Did it bind where you asked?

This is the check from the warning above. It counts, for the best design, how
many contacts fall on your chosen hotspot and its neighbours versus elsewhere.


In [ ]:
#@title 🔍 Check the binding site, and look at it
import glob, subprocess, sys
import numpy as np
try:
    import py3Dmol
except ModuleNotFoundError:          # fresh runtime, or cell 1 not run
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=True)
    import py3Dmol

f = sorted(glob.glob(f'{OUT}/final_ranked_designs/final_*_designs/*.cif'))
if not f:
    print('No structures found.')
else:
    path = f[0]
    hdr, rows, inloop = [], [], False
    for ln in open(path):
        if ln.startswith('_atom_site.'):
            hdr.append(ln.strip().split('.')[1]); inloop = True; continue
        if inloop and ln.startswith(('ATOM', 'HETATM')): rows.append(ln.split())
        elif inloop and ln.startswith('#') and rows: break
    ci, si, xi = (hdr.index('label_asym_id'), hdr.index('label_seq_id'),
                  hdr.index('Cartn_x'))
    tgt = [r for r in rows if r[ci] == keep_chain]
    bnd = [r for r in rows if r[ci] != keep_chain]
    T = np.array([[float(r[xi]), float(r[xi+1]), float(r[xi+2])] for r in tgt])
    B = np.array([[float(r[xi]), float(r[xi+1]), float(r[xi+2])] for r in bnd])
    d = np.linalg.norm(B[:, None, :] - T[None, :, :], axis=-1)
    touched = sorted({int(tgt[j][si]) for j in np.where((d < 5.0).any(axis=0))[0]})

    # map back to the PDB's own numbering
    touched_pdb = sorted({resids[i-1] for i in touched if 0 < i <= len(resids)})
    near = [r for r in touched_pdb if abs(r - hotspot_residue) <= 5]
    print(f'binder contacts {len(touched_pdb)} target residues: {touched_pdb}')
    print(f'you asked for residue {hotspot_residue}')
    if hotspot_residue in touched_pdb:
        print('  ✅ it bound your hotspot directly')
    elif near:
        print(f'  ~ it landed nearby ({near}) but not on your residue')
    else:
        print('  ❌ it ignored your hotspot and bound somewhere else entirely')
        print('     This is common. The hotspot is a preference, not a constraint.')

    view = py3Dmol.view(width=760, height=560)
    view.addModel(open(path).read(), 'cif')
    binder_chain = next(c for c in {r[ci] for r in rows} if c != keep_chain)
    view.setStyle({'chain': keep_chain}, {'cartoon': {'color': 'lightgrey'}})
    view.setStyle({'chain': binder_chain}, {'cartoon': {'color': 'orange'}})
    view.addStyle({'chain': keep_chain, 'resi': [str(hotspot_residue)]},
                  {'stick': {'colorscheme': 'redCarbon'}})
    view.setBackgroundColor('white'); view.zoomTo(); view.show()
    print('\ngrey = target, orange = your binder, red sticks = your hotspot')


In [ ]:
#@title 💾 Download your results
import subprocess
from google.colab import files
subprocess.run(f'zip -qr /content/{run_name}.zip {OUT}', shell=True)
files.download(f'/content/{run_name}.zip')


---
## Things to try

- **Run the same hotspot twice.** The model is a sampler — you will get two
  different binders. How different?
- **Pick a hotspot on the opposite face** and see whether the binder follows you
  there, or goes back to wherever it preferred the first time.
- **Change the binder length.** Short binders bury less surface; long ones are
  harder to make.
- **Sort by `liability_num_violations` instead of ipTM.** Does the ranking change?

## For the discussion

Bring your best design's **ipTM**, its **liabilities**, and whether it bound
where you asked. Be ready to say whether you would spend money making it.

---

Published campaigns generate **10,000–60,000** designs and test ~150. You ran
four. Keep that in proportion when you look at your numbers.

BoltzGen: Stark et al., 2025. MIT licence.
